# Homework 9: Predicting Automobile Prices
Jacob A. Fericy

In this homework, we want to build a supervised learning workflow in PySpark using the UCI machine learning library with an open-source automobile dataset, comparing three model classes with cross validation results, and then selecting an overall best model.

**Dataset:** UCI Automobile dataset  
**Task:** Regression on `price`  
**Metric:** Root Mean Squared Error (RMSE)

This notebook uses the UCI Automobile data set. UCI describes the data as a multivariate automobile dataset with many instances, features, a regression task, and it also includes several missing values. The repo mentions that `price` can be used as a target variable and that missing values are encoded as `"?"`.

## Modeling Outline
In this homework, I want to predict automobile **price** from the vehicle's technical and categorical characteristics. The reason for the regression is that `price` is numeric, has a meaningful business interpretation, and is one of the variables UCI identifies as usable as a target.

We want to use the following models: Linear Regression, Decision Tree Regression and Random Forest. For LR, a parametric baseline assumes the response is a linear function of the predictors. This is easy to interpret and often works well when relationships are approximately additive, so it could work in this case. For DT, it is a non-linear model that recursively splits the feature space into similar distributed regions. It captures interactions and threshold effects without requiring manual specification. For random forest, it is an ensemble of decision trees fit on bootstrap samples and random subsets of predictors. This usually reduces variance relative to a single tree and often performs better on structured tabular data, so it should fit the data well.

We want to evaluate **RMSE** as the primary metric. RMSE is measured in the same units as the response variable, which makes it interpretable in dollars. Lower RMSE indicates better predictive performance.

In [31]:
# ran into a local error, so here is an import if you get the bug locally. 
# %pip install pyspark

import warnings
warnings.filterwarnings("ignore")

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer,
    OneHotEncoder,
    Imputer,
    VectorAssembler,
    StandardScaler,
    PolynomialExpansion
)
from pyspark.ml.regression import LinearRegression, DecisionTreeRegressor, RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

import pandas as pd
import numpy as np
import urllib.request
from pathlib import Path

## Create Spark Instance

In [32]:
spark = (
    SparkSession.builder
    .appName("HW9_Auto")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark

## Load Data

UCI provides the Automobile dataset and documents the attributes, missingness, and target variable. The data file is `imports-85.data`. To make the notebook reproducible, the code below first checks whether the data file exists locally. If not, it downloads the file from UCI. This may have been lacking in previous HWs. Apologies. 

In [33]:
columns = [
    "symboling", "normalized_losses", "make", "fuel_type", "aspiration",
    "num_doors", "body_style", "drive_wheels", "engine_location",
    "wheel_base", "length", "width", "height", "curb_weight",
    "engine_type", "num_cylinders", "engine_size", "fuel_system",
    "bore", "stroke", "compression_ratio", "horsepower", "peak_rpm",
    "city_mpg", "highway_mpg", "price"
]

data_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/autos/imports-85.data"
local_file = Path("imports-85.data")

if not local_file.exists():
    urllib.request.urlretrieve(data_url, local_file)

raw = (
    spark.read
    .option("header", False)
    .option("inferSchema", False)
    .csv(str(local_file))
)

for i, col_name in enumerate(columns):
    raw = raw.withColumnRenamed(f"_c{i}", col_name)

raw.printSchema()
raw.show(5, truncate=False)

root
 |-- symboling: string (nullable = true)
 |-- normalized_losses: string (nullable = true)
 |-- make: string (nullable = true)
 |-- fuel_type: string (nullable = true)
 |-- aspiration: string (nullable = true)
 |-- num_doors: string (nullable = true)
 |-- body_style: string (nullable = true)
 |-- drive_wheels: string (nullable = true)
 |-- engine_location: string (nullable = true)
 |-- wheel_base: string (nullable = true)
 |-- length: string (nullable = true)
 |-- width: string (nullable = true)
 |-- height: string (nullable = true)
 |-- curb_weight: string (nullable = true)
 |-- engine_type: string (nullable = true)
 |-- num_cylinders: string (nullable = true)
 |-- engine_size: string (nullable = true)
 |-- fuel_system: string (nullable = true)
 |-- bore: string (nullable = true)
 |-- stroke: string (nullable = true)
 |-- compression_ratio: string (nullable = true)
 |-- horsepower: string (nullable = true)
 |-- peak_rpm: string (nullable = true)
 |-- city_mpg: string (nullable = t

## Clean Data

According to UCI, missing values are represented by `"?"`, while several fields contain missing data, including `price`, `normalized-losses`, `bore`, `stroke`, `horsepower`, and `peak-rpm`.

Below are the cleaing steps:
- Replace `"?"` with null
- Cast numeric fields to numeric type
- Drop rows with missing response (`price`)
- Keep the remaining missing predictors for imputation inside the modeling pipeline

In [34]:
df = raw

# Replace "?" with null in every column
for c in df.columns:
    df = df.withColumn(c, F.when(F.col(c) == "?", None).otherwise(F.col(c)))

numeric_cols = [
    "symboling", "normalized_losses", "wheel_base", "length", "width", "height",
    "curb_weight", "engine_size", "bore", "stroke", "compression_ratio",
    "horsepower", "peak_rpm", "city_mpg", "highway_mpg", "price"
]

for c in numeric_cols:
    df = df.withColumn(c, F.col(c).cast(DoubleType()))

# Drop rows with missing price because price is the response
df = df.filter(F.col("price").isNotNull())

print("Rows after dropping missing price:", df.count())
df.printSchema()

Rows after dropping missing price: 201
root
 |-- symboling: double (nullable = true)
 |-- normalized_losses: double (nullable = true)
 |-- make: string (nullable = true)
 |-- fuel_type: string (nullable = true)
 |-- aspiration: string (nullable = true)
 |-- num_doors: string (nullable = true)
 |-- body_style: string (nullable = true)
 |-- drive_wheels: string (nullable = true)
 |-- engine_location: string (nullable = true)
 |-- wheel_base: double (nullable = true)
 |-- length: double (nullable = true)
 |-- width: double (nullable = true)
 |-- height: double (nullable = true)
 |-- curb_weight: double (nullable = true)
 |-- engine_type: string (nullable = true)
 |-- num_cylinders: string (nullable = true)
 |-- engine_size: double (nullable = true)
 |-- fuel_system: string (nullable = true)
 |-- bore: double (nullable = true)
 |-- stroke: double (nullable = true)
 |-- compression_ratio: double (nullable = true)
 |-- horsepower: double (nullable = true)
 |-- peak_rpm: double (nullable = tr

## Data Summary

UCI reports 205 observations in the original data and missing values in multiple columns. After removing rows with missing `price`, the working analysis sample is slightly smaller.

In [35]:
# Missing values by column
missing_summary = (
    df.select([
        F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in df.columns
    ])
)

missing_pd = missing_summary.toPandas().T.reset_index()
missing_pd.columns = ["column", "missing_count"]
missing_pd = missing_pd.sort_values("missing_count", ascending=False)
missing_pd

,column,missing_count
1,normalized_losses,37
19,stroke,4
18,bore,4
22,peak_rpm,2
5,num_doors,2
21,horsepower,2
0,symboling,0
14,engine_type,0
24,highway_mpg,0
23,city_mpg,0


In [36]:
# Summ of response variable
df.select("price").summary().show()

+-------+------------------+
|summary|             price|
+-------+------------------+
|  count|               201|
|   mean|13207.129353233831|
| stddev| 7947.066341939271|
|    min|            5118.0|
|    25%|            7775.0|
|    50%|           10295.0|
|    75%|           16500.0|
|    max|           45400.0|
+-------+------------------+



In [37]:
# Mean price by make
(
    df.groupBy("make")
      .agg(
          F.count("*").alias("n"),
          F.round(F.avg("price"), 2).alias("avg_price")
      )
      .orderBy(F.desc("avg_price"))
      .show(25, truncate=False)
)

+-------------+---+---------+
|make         |n  |avg_price|
+-------------+---+---------+
|jaguar       |3  |34600.0  |
|mercedes-benz|8  |33647.0  |
|porsche      |4  |31400.5  |
|bmw          |8  |26118.75 |
|volvo        |11 |18063.18 |
|audi         |6  |17859.17 |
|mercury      |1  |16503.0  |
|alfa-romero  |3  |15498.33 |
|peugot       |11 |15489.09 |
|saab         |6  |15223.33 |
|mazda        |17 |10652.88 |
|nissan       |18 |10415.67 |
|volkswagen   |12 |10077.5  |
|toyota       |32 |9885.81  |
|renault      |2  |9595.0   |
|mitsubishi   |13 |9239.77  |
|isuzu        |2  |8916.5   |
|subaru       |12 |8541.25  |
|honda        |13 |8184.69  |
|plymouth     |7  |7963.43  |
|dodge        |9  |7875.44  |
|chevrolet    |3  |6007.0   |
+-------------+---+---------+



## Train/test split

I use an 80/20 split with a fixed seed for reproducibility.

In [38]:
train_df, test_df = df.randomSplit([0.8, 0.2], seed = 316)

print("Training rows:", train_df.count())
print("Testing rows: ", test_df.count())

Training rows: 161
Testing rows:  40


##  Predictors and Data Wrangling

The response is `price`. All remaining variables are candidate predictors. I used StringIndexer and OneHotEncoder encoding. 

I also imputed with the median using `Imputer`. This allows the transformations tied to each model to avoid leakage from the test set.

At least one pipeline must contain four or more transformations before the estimator. The linear regression pipeline below includes:
1. Imputer  
2. StringIndexer  
3. OneHotEncoder  
4. VectorAssembler  
5. StandardScaler  
6. PolynomialExpansion  

In [39]:
target_col = "price"

categorical_cols = [
    "make", "fuel_type", "aspiration", "num_doors", "body_style",
    "drive_wheels", "engine_location", "engine_type", "num_cylinders",
    "fuel_system"
]

numeric_feature_cols = [
    "symboling", "normalized_losses", "wheel_base", "length", "width", "height",
    "curb_weight", "engine_size", "bore", "stroke", "compression_ratio",
    "horsepower", "peak_rpm", "city_mpg", "highway_mpg"
]

indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in categorical_cols
]

encoders = [
    OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_ohe", handleInvalid="keep")
    for c in categorical_cols
]

imputer = Imputer(
    inputCols=numeric_feature_cols,
    outputCols=[f"{c}_imp" for c in numeric_feature_cols],
    strategy="median"
)

assembled_inputs = [f"{c}_imp" for c in numeric_feature_cols] + [f"{c}_ohe" for c in categorical_cols]
assembler = VectorAssembler(inputCols=assembled_inputs, outputCol="features_raw")

## Build model pipelines

I compare three distinct model classes:
- Linear Regression
- Decision Tree Regressor
- Random Forest Regressor

The linear regression pipeline gets extra feature processing because linear models are more sensitive to scaling and can benefit from polynomial terms. Tree methods do not need scaling.

In [40]:
#pipe1 LR
scaler = StandardScaler(inputCol="features_raw", outputCol="features_scaled", withMean=False, withStd=True)
poly = PolynomialExpansion(inputCol="features_scaled", outputCol="features", degree=2)
lr = LinearRegression(featuresCol="features", labelCol=target_col, maxIter=200)

lr_pipeline = Pipeline(stages=[imputer] + indexers + encoders + [assembler, scaler, poly, lr])

#pipe2 DT
dt = DecisionTreeRegressor(featuresCol="features_raw", labelCol=target_col, seed=316)
dt_pipeline = Pipeline(stages=[imputer] + indexers + encoders + [assembler, dt])

#pipe3 RF
rf = RandomForestRegressor(featuresCol="features_raw", labelCol=target_col, seed=316)
rf_pipeline = Pipeline(stages=[imputer] + indexers + encoders + [assembler, rf])

## Cross Validation Setup

I use 5-fold cross validation and RMSE as the selection criterion. Lower RMSE is better.

In [41]:
rmse_evaluator = RegressionEvaluator(labelCol=target_col, predictionCol="prediction", metricName="rmse")
r2_evaluator = RegressionEvaluator(labelCol=target_col, predictionCol="prediction", metricName="r2")

lr_grid = (
    ParamGridBuilder()
    .addGrid(lr.regParam, [0.0, 0.01, 0.1])
    .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0])
    .build()
)

dt_grid = (
    ParamGridBuilder()
    .addGrid(dt.maxDepth, [3, 5, 8, 12])
    .addGrid(dt.minInstancesPerNode, [1, 3, 5])
    .build()
)

rf_grid = (
    ParamGridBuilder()
    .addGrid(rf.numTrees, [20, 50])
    .addGrid(rf.maxDepth, [3, 5])
    .addGrid(rf.minInstancesPerNode, [1, 3])
    .build()
)

lr_cv = CrossValidator(
    estimator=lr_pipeline,
    estimatorParamMaps=lr_grid,
    evaluator=rmse_evaluator,
    numFolds=5,
    seed=42
)

dt_cv = CrossValidator(
    estimator=dt_pipeline,
    estimatorParamMaps=dt_grid,
    evaluator=rmse_evaluator,
    numFolds=5,
    seed=42
)

rf_cv = CrossValidator(
    estimator=rf_pipeline,
    estimatorParamMaps=rf_grid,
    evaluator=rmse_evaluator,
    numFolds=2,
    seed=42
)

## Model Fit: Three Types

In this section, we fit each model type with the training data that we have. 

In [ ]:
lr_cv_model = lr_cv.fit(train_df)
dt_cv_model = dt_cv.fit(train_df)
rf_cv_model = rf_cv.fit(train_df)

## Cross Validated Training

Spark stores the average cross-validated metric values for each tuning grid. Because the evaluator is RMSE, lower values indicate better models.

In [ ]:
cv_results = pd.DataFrame({
    "model": ["Linear Regression", "Decision Tree", "Random Forest"],
    "best_cv_rmse": [
        float(np.min(lr_cv_model.avgMetrics)),
        float(np.min(dt_cv_model.avgMetrics)),
        float(np.min(rf_cv_model.avgMetrics))
    ]
}).sort_values("best_cv_rmse")

cv_results

## Best Models

Cross validation chooses the best tuned version of each model class using the training data only. The final comparison is then performed on the held-out test set so we can ensure more accurate testing and lack of bias in the testing environment. 

In [ ]:
lr_pred = lr_cv_model.bestModel.transform(test_df)
dt_pred = dt_cv_model.bestModel.transform(test_df)
rf_pred = rf_cv_model.bestModel.transform(test_df)

test_results = pd.DataFrame({
    "model": ["Linear Regression", "Decision Tree", "Random Forest"],
    "test_rmse": [
        rmse_evaluator.evaluate(lr_pred),
        rmse_evaluator.evaluate(dt_pred),
        rmse_evaluator.evaluate(rf_pred)
    ],
    "test_r2": [
        r2_evaluator.evaluate(lr_pred),
        r2_evaluator.evaluate(dt_pred),
        r2_evaluator.evaluate(rf_pred)
    ]
}).sort_values("test_rmse")

test_results

## Best overall model

The best overall model is the one with the **lowest test RMSE** so in this case that is the random forest. This is harder on the intepretation side, however, statistically it is more sound.

In [ ]:
best_row = test_results.iloc[0]

print(f"Best overall model: {best_row['model']}")
print(f"Test RMSE: {best_row['test_rmse']:.2f}")
print(f"Test R^2: {best_row['test_r2']:.4f}")

## Interpretation and discussion

Each model allowed us to intepret different outcomes. The linear regression added an interpretable baseline and shows how far a simple approach can go after data wrangling. The decision tree approach  captures non-linear structure and interactions automatically, but a single tree can be unstable and prone to overfitting. The random forest usually offers the best balance of flexibility and generalization because averaging many trees reduces variance.

On structured tabular data like automobile attributes, a random forest often performs best because relationships between specifications and price are rarely purely linear. Still, the notebook is set up to let the data decide rather than assuming that in advance.

There are some limitations to the model. The dataset is small and some of the predictors contains missing values. Prices come from a specific historical context, so this is a prediction exercise, not a modern market valuation model.

Those limitations are all documented by UCI or obvious from the sample size and feature structure.

In [ ]:
spark.stop()